In [1]:
!pip install wandb

Install UnSloth

In [2]:
!pip install unsloth

In [3]:
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-req-build-37rtsj2k
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-req-build-37rtsj2k
  Resolved https://github.com/unslothai/unsloth.git to commit 7f27d8eadd5e1437eb0d5cdab738e31014657179
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.7.6-py3-none-any.whl size=39207017 sha256=891165051d4d4ad132f50fe4201d854c841aaeb07432159436ffceed9ecf0d8e
  Stored in directory: /tmp/pip-ephem-wheel-cache-hrdhpnqi/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2026.7.6
    Uninstalling unsloth-2026.7.6:
      Successfully uninstalled unsloth-2026.7.6


In [4]:
!pip install transformers
!pip install datasets
!pip install trl

In [5]:
!pip install --upgrade --no-deps torchaudio torchvision --extra-index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu130


Loading the model

In [6]:
#!pip install --upgrade "torchvision>=0.28.0"

In [7]:

from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0729 21:42:33.232000 33719 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0729 21:42:33.279000 33719 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.12/dist-packages/unsloth/import_fixes.py:1208: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


In [8]:
model, tokenizer = FastLanguageModel.from_pretrained(model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = 2048, dtype = None, load_in_4bit = True, )

==((====))==  Unsloth 2026.7.6: Fast Llama patching. Transformers: 5.14.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.13.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/deepseek-r1-distill-llama-8b-unsloth-bnb-4bit as a legacy tokenizer.


Loading Dataset

In [22]:
from datasets import load_dataset
dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT","en", split = "train[0:500]",trust_remote_code=True)
dataset = dataset.map(formatting_prompts_func, batched = True,)
dataset["text"][0]


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'FreedomIntelligence/medical-o1-reasoning-SFT' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'FreedomIntelligence/medical-o1-reasoning-SFT' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


"Below is an instruction that describes a task, paired with an input that provides further context. \nWrite a response that appropriately completes the request. \nBefore answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.\n\n### Instruction:\nYou are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. \nPlease answer the following medical question. \n\n### Question:\nGiven the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?\n\n### Response:\n<think>\nOkay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg i

Model Inference before Fine-Tuning

In [10]:
prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Please answer the following medical question.

### Question:
{}

###Response:
<think>{}"""

In [25]:
question = "A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings, what would cystometry most likely reveal about her residual volume and detrusor contractions?"


FastLanguageModel.for_inference(model)  # Unsloth has 2x faster inference!
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)
#print(response[0].split("### Response:")[1])

Both `max_new_tokens` (=1200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [28]:
print(response[0].split("###Response"))

['<｜begin of sentence｜>Belowisaninstructionthatdescribesatask,pairedwithaninputthatprovidesfurthercontext.Writearesponsethatappropriatelycompletestherequest.Beforeanswering,thinkcarefullyaboutthequestionandcreateastep-by-stepchainofthoughtstoensurealogicalandaccurateresponse.###Instruction:Youareamedicalexpertwithadvancedknowledgeinclinicalreasoning,diagnostics,andtreatmentplanning.Pleaseanswerthefollowingmedicalquestion.###Question:A61-year-oldwomanwithalonghistoryofinvoluntaryurinelossduringactivitieslikecoughingorsneezingbutnoleakageatnightundergoesagynecologicalexamandQ-tiptest.Basedonthesefindings,whatwouldcystometrymostlikelyrevealaboutherresidualvolumeanddetrusorcontractions?', ":<think>ĊOkay,ĠsoĠIĠhaveĠthisĠmedicalĠquestionĠtoĠanswer,ĠandĠIĠneedĠtoĠmakeĠsureĠIĠgetĠitĠright.ĠLetĠmeĠstartĠbyĠunderstandingĠwhat'sĠbeingĠaskedĠhere.ĊĊTheĠpatientĠisĠaĠ61-year-oldĠwomanĠwho'sĠbeenĠdealingĠwithĠinvoluntaryĠurineĠlossĠduringĠactivitiesĠlikeĠcoughingĠorĠsneezing,ĠbutĠsheĠdoesn'tĠleakĠatĠ

In [29]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for very long context
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

Unsloth: Already have LoRA adapters! We shall skip this step.


Train dataset

In [30]:
train_prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Please answer the following medical question.

### Question:
{}

### Response:
<think>
{}
</think>
{}"""

Tokenize

In [31]:
EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN


def formatting_prompts_func(examples):
    inputs = examples["Question"]
    cots = examples["Complex_CoT"]
    outputs = examples["Response"]
    texts = []
    for input, cot, output in zip(inputs, cots, outputs):
        text = train_prompt_style.format(input, cot, output) + EOS_TOKEN
        texts.append(text)
    return {
        "text": texts,
    }

Load dataset

In [15]:
from datasets import load_dataset
dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT","en", split = "train[0:500]",trust_remote_code=True)
dataset = dataset.map(formatting_prompts_func, batched = True,)
dataset["text"][0]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'FreedomIntelligence/medical-o1-reasoning-SFT' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'FreedomIntelligence/medical-o1-reasoning-SFT' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


"Below is an instruction that describes a task, paired with an input that provides further context. \nWrite a response that appropriately completes the request. \nBefore answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.\n\n### Instruction:\nYou are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. \nPlease answer the following medical question. \n\n### Question:\nGiven the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?\n\n### Response:\n<think>\nOkay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg i

In [16]:
"Below is an instruction that describes a task, paired with an input that provides further context. \nWrite a response that appropriately completes the request. \nBefore answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.\n\n### Instruction:\nYou are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. \nPlease answer the following medical question. \n\n### Question:\nA 61-year-old woman with a long history of involuntary urine loss during activities like coughing or sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings, what would cystometry most likely reveal about her residual volume and detrusor contractions?\n\n### Response:\n<think>\nOkay, let's think about this step by step. There's a 61-year-old woman here who's been dealing with involuntary urine leakages whenever she's doing something that ups her abdominal pressure like coughing or sneezing. This sounds a lot like stress urinary incontinence to me. Now, it's interesting that she doesn't have any issues at night; she isn't experiencing leakage while sleeping. This likely means her bladder's ability to hold urine is fine when she isn't under physical stress. Hmm, that's a clue that we're dealing with something related to pressure rather than a bladder muscle problem. \n\nThe fact that she underwent a Q-tip test is intriguing too. This test is usually done to assess urethral mobility. In stress incontinence, a Q-tip might move significantly, showing urethral hypermobility. This kind of movement often means there's a weakness in the support structures that should help keep the urethra closed during increases in abdominal pressure. So, that's aligning well with stress incontinence.\n\nNow, let's think about what would happen during cystometry. Since stress incontinence isn't usually about sudden bladder contractions, I wouldn't expect to see involuntary detrusor contractions during this test. Her bladder isn't spasming or anything; it's more about the support structure failing under stress. Plus, she likely empties her bladder completely because stress incontinence doesn't typically involve incomplete emptying. So, her residual volume should be pretty normal. \n\nAll in all, it seems like if they do a cystometry on her, it will likely show a normal residual volume and no involuntary contractions. Yup, I think that makes sense given her symptoms and the typical presentations of stress urinary incontinence.\n</think>\nCystometry in this case of stress urinary incontinence would most likely reveal a normal post-void residual volume, as stress incontinence typically does not involve issues with bladder emptying. Additionally, since stress urinary incontinence is primarily related to physical exertion and not an overactive bladder, you would not expect to see any involuntary detrusor contractions during the test.<｜end▁of▁sentence｜>"

"Below is an instruction that describes a task, paired with an input that provides further context. \nWrite a response that appropriately completes the request. \nBefore answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.\n\n### Instruction:\nYou are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. \nPlease answer the following medical question. \n\n### Question:\nA 61-year-old woman with a long history of involuntary urine loss during activities like coughing or sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings, what would cystometry most likely reveal about her residual volume and detrusor contractions?\n\n### Response:\n<think>\nOkay, let's think about this step by step. There's a 61-year-old woman here who's been dealing with involuntary urine leakages whenever she's doing something that ups her ab

Setting the model

In [36]:
from unsloth import is_bfloat16_supported
from trl import SFTTrainer
from trl import SFTConfig

In [18]:
!rm -rf /content/unsloth_compiled_cache
!pip install --upgrade --no-deps unsloth unsloth_zoo

In [41]:
args = SFTConfig(
    output_dir = "outputs",
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = True, # Set True for fast packing
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
)

In [39]:
from transformers import TrainingArguments

In [42]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = args,
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


Model Training

In [43]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 215 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.924524
2,2.809340
3,2.922608
4,2.831500
5,2.769670
6,2.602782
7,2.329188
8,2.253541
9,2.148944
10,2.134727


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


save the result

In [50]:
import wandb
from google.colab import userdata
wb_token = userdata.get('WeightsAndBiases')
wandb.login(key=wb_token)
run = wandb.init(
    project='Fine-tune-DeepSeek-R1-Distill-Llama-8B on Medical COT Dataset',
    job_type="training",
    anonymous="allow"
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: eeshairfan1907 (eeshairfan1907-nuces) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


In [51]:
wandb.finish()

after fine tuning

In [54]:
question = "A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings, what would cystometry most likely reveal about her residual volume and detrusor contractions?"


FastLanguageModel.for_inference(model)  # Unsloth has 2x faster inference!
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)

Both `max_new_tokens` (=1200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [57]:
print(response[0].split("### Response:")[0])

<｜begin of sentence｜>Belowisaninstructionthatdescribesatask,pairedwithaninputthatprovidesfurthercontext.Writearesponsethatappropriatelycompletestherequest.Beforeanswering,thinkcarefullyaboutthequestionandcreateastep-by-stepchainofthoughtstoensurealogicalandaccurateresponse.###Instruction:Youareamedicalexpertwithadvancedknowledgeinclinicalreasoning,diagnostics,andtreatmentplanning.Pleaseanswerthefollowingmedicalquestion.###Question:A61-year-oldwomanwithalonghistoryofinvoluntaryurinelossduringactivitieslikecoughingorsneezingbutnoleakageatnightundergoesagynecologicalexamandQ-tiptest.Basedonthesefindings,whatwouldcystometrymostlikelyrevealaboutherresidualvolumeanddetrusorcontractions?###Response:<think>Alright,let'sstartbythinkingaboutthispatient'shistory.She's61andhasbeenexperiencingsomekindofinvoluntaryurinelossevercoughingorsneezing,whichsoundslikeurinaryincontinence.Butit'snotjustleakingatnight,sowe'relookingatmorethanjustnoclassicsleepincontinence.Let'sconsiderthistestshehad,thegyneco

In [58]:
question = "A 59-year-old man presents with a fever, chills, night sweats, and generalized fatigue, and is found to have a 12 mm vegetation on the aortic valve. Blood cultures indicate gram-positive, catalase-negative, gamma-hemolytic cocci in chains that do not grow in a 6.5% NaCl medium. What is the most likely predisposing factor for this patient's condition?"

inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)

Both `max_new_tokens` (=1200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [60]:
print(response[0].split("### Response:")[0])

<｜begin of sentence｜>Belowisaninstructionthatdescribesatask,pairedwithaninputthatprovidesfurthercontext.Writearesponsethatappropriatelycompletestherequest.Beforeanswering,thinkcarefullyaboutthequestionandcreateastep-by-stepchainofthoughtstoensurealogicalandaccurateresponse.###Instruction:Youareamedicalexpertwithadvancedknowledgeinclinicalreasoning,diagnostics,andtreatmentplanning.Pleaseanswerthefollowingmedicalquestion.###Question:A59-year-oldmanpresentswithafever,chills,nightsweats,andgeneralizedfatigue,andisfoundtohavea12mmvegetationontheaorticvalve.Bloodculturesindicategram-positive,catalase-negative,gamma-hemolyticcocciinchainsthatdonotgrowina6.5%NaClmedium.Whatisthemostlikelypredisposingfactorforthispatient'scondition?###Response:<think>Alright,let'sthinkaboutthis.Thepatientis59andhasafewclassicsigns:afever,feversweats,andfeelingtiredallthetime.Thatsoundslikeitcouldbeinfectious,butlet'stalkaboutthevegetationonhisheartĠvalve.It's12mm,andthat'ssomethingtorememberbecauseit'squitebig.

Saving the model locally

In [62]:
new_model_online = "Medical-FineTuning"
new_model_local = "Medical-FineTuning_DeepSeek"
model.save_pretrained(new_model_local) # Local saving
tokenizer.save_pretrained(new_model_local)

Unsloth: Restored added_tokens_decoder metadata in Medical-FineTuning_DeepSeek/tokenizer_config.json.


('Medical-FineTuning_DeepSeek/tokenizer_config.json',
 'Medical-FineTuning_DeepSeek/chat_template.jinja',
 'Medical-FineTuning_DeepSeek/tokenizer.json')

In [64]:
model.save_pretrained("Medical-FineTuning")
tokenizer.save_pretrained("Medical-FineTuning")

Unsloth: Restored added_tokens_decoder metadata in Medical-FineTuning/tokenizer_config.json.


('Medical-FineTuning/tokenizer_config.json',
 'Medical-FineTuning/chat_template.jinja',
 'Medical-FineTuning/tokenizer.json')